# Setup

In [1]:
%pip install -q mediapipe opencv-python


[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2
import mediapipe as mp
import time
import numpy as np
import argparse
import os
from collections import defaultdict

# Detectores

In [3]:
class HandDetector:
    def __init__(self, static_mode=False, max_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5):
        """
        Initialize the hand detector with MediaPipe

        Args:
            static_mode (bool): If set to False, the solution treats the input images as a video stream
            max_hands (int): Maximum number of hands to detect
            min_detection_confidence (float): Minimum confidence value for hand detection to be considered successful
            min_tracking_confidence (float): Minimum confidence value for the hand landmarks to be considered tracked successfully
        """
        self.results = None
        self.static_mode = static_mode
        self.max_hands = max_hands
        self.min_detection_confidence = min_detection_confidence
        self.min_tracking_confidence = min_tracking_confidence

        # Initialize MediaPipe Hands solution
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=self.static_mode,
            max_num_hands=self.max_hands,
            min_detection_confidence=self.min_detection_confidence,
            min_tracking_confidence=self.min_tracking_confidence
        )

        # Initialize drawing utility
        self.mp_draw = mp.solutions.drawing_utils
        self.mp_drawing_styles = mp.solutions.drawing_styles

    def find_hands(self, img, draw=True):
        """
        Detect hands in an image and draw landmarks if specified

        Args:
            img (numpy.ndarray): Input image
            draw (bool): If True, draws the landmarks on the image

        Returns:
            numpy.ndarray: Image with or without drawings
            list: List of detected hands
        """
        # Convert BGR image to RGB
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Process the image and find hands
        self.results = self.hands.process(img_rgb)

        # List to store all hand information
        all_hands = []

        # Check if hands are detected
        if self.results.multi_hand_landmarks:
            for hand_idx, hand_landmarks in enumerate(self.results.multi_hand_landmarks):
                # Draw landmarks if specified
                if draw:
                    self.mp_draw.draw_landmarks(
                        img,
                        hand_landmarks,
                        self.mp_hands.HAND_CONNECTIONS,
                        self.mp_drawing_styles.get_default_hand_landmarks_style(),
                        self.mp_drawing_styles.get_default_hand_connections_style()
                    )

                # Extract landmark positions
                hand_info = {}
                landmarks = []
                for lm_id, lm in enumerate(hand_landmarks.landmark):
                    h, w, c = img.shape
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    landmarks.append((lm_id, cx, cy))

                # Get hand type (Left or Right)
                if self.results.multi_handedness:
                    hand_type = self.results.multi_handedness[hand_idx].classification[0].label
                    hand_info["type"] = hand_type

                hand_info["landmarks"] = landmarks
                all_hands.append(hand_info)

        return img, all_hands

    def find_position(self, img, hand_index=0, draw=True):
        """
        Find the position of landmarks for a specific hand

        Args:
            img (numpy.ndarray): Input image
            hand_index (int): Index of the hand to find positions for
            draw (bool): If True, draws circles at landmark positions

        Returns:
            list: List of landmark positions with [id, x, y]
        """
        landmark_list = []

        if self.results.multi_hand_landmarks:
            if len(self.results.multi_hand_landmarks) > hand_index:
                my_hand = self.results.multi_hand_landmarks[hand_index]

                for lm_id, lm in enumerate(my_hand.landmark):
                    h, w, c = img.shape
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    landmark_list.append([lm_id, cx, cy])

                    if draw:
                        cv2.circle(img, (cx, cy), 5, (255, 0, 255), cv2.FILLED)

        return landmark_list

In [4]:
class PersonTracker:
    def __init__(self):
        self.next_id = 0
        self.persons = {}  # id -> bounding box
        self.frames_missing = {}  # id -> count of frames the person was missing
        self.confidence_scores = {}  # id -> confidence score
        self.max_missing_frames = 30  # If a person is missing for this many frames, remove them

    def update(self, new_boxes, scores=None):
        """
        Update person tracking with new bounding boxes

        Args:
            new_boxes: List of (x1, y1, x2, y2) bounding boxes
            scores: Optional list of confidence scores

        Returns:
            Dict mapping person_id to bounding box
        """
        if scores is None:
            scores = [1.0] * len(new_boxes)

        # If no existing persons, assign IDs to all new boxes
        if not self.persons:
            for i, box in enumerate(new_boxes):
                if i < len(scores):
                    self.persons[self.next_id] = box
                    self.frames_missing[self.next_id] = 0
                    self.confidence_scores[self.next_id] = scores[i]
                    self.next_id += 1
            return self.persons.copy()

        # Calculate IoU between existing persons and new boxes
        assignments = {}  # new_box_idx -> person_id
        for person_id, person_box in self.persons.items():
            for i, new_box in enumerate(new_boxes):
                if i in assignments:
                    continue  # This box is already assigned

                # Calculate IoU between boxes
                iou = self._calculate_iou(person_box, new_box)

                # If IoU is high enough, assign this box to this person
                if iou > 0.3:  # Threshold can be adjusted
                    assignments[i] = person_id
                    self.persons[person_id] = new_box
                    self.frames_missing[person_id] = 0
                    if i < len(scores):
                        # Update confidence as moving average
                        old_conf = self.confidence_scores[person_id]
                        new_conf = scores[i]
                        self.confidence_scores[person_id] = 0.7 * old_conf + 0.3 * new_conf

        # Mark all unassigned persons as missing
        assigned_persons = set(assignments.values())
        for person_id in list(self.persons.keys()):
            if person_id not in assigned_persons:
                self.frames_missing[person_id] += 1

        # Remove persons missing for too long
        for person_id in list(self.persons.keys()):
            if self.frames_missing[person_id] > self.max_missing_frames:
                del self.persons[person_id]
                del self.frames_missing[person_id]
                del self.confidence_scores[person_id]

        # Assign IDs to new boxes without assignments
        for i, new_box in enumerate(new_boxes):
            if i not in assignments:
                self.persons[self.next_id] = new_box
                self.frames_missing[self.next_id] = 0
                self.confidence_scores[self.next_id] = scores[i] if i < len(scores) else 1.0
                assignments[i] = self.next_id
                self.next_id += 1

        return self.persons.copy()

    def get_high_confidence_persons(self, threshold=0.6):
        """Return only persons with confidence above threshold"""
        result = {}
        for person_id, box in self.persons.items():
            if self.confidence_scores.get(person_id, 0) >= threshold:
                result[person_id] = box
        return result

    def _calculate_iou(self, box1, box2):
        """Calculate Intersection over Union between two bounding boxes"""
        x1_1, y1_1, x2_1, y2_1 = box1
        x1_2, y1_2, x2_2, y2_2 = box2

        # Calculate intersection area
        x_left = max(x1_1, x1_2)
        y_top = max(y1_1, y1_2)
        x_right = min(x2_1, x2_2)
        y_bottom = min(y2_1, y2_2)

        if x_right < x_left or y_bottom < y_top:
            return 0.0

        intersection_area = (x_right - x_left) * (y_bottom - y_top)

        # Calculate union area
        box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
        box2_area = (x2_2 - x1_2) * (y2_2 - y1_2)
        union_area = box1_area + box2_area - intersection_area

        return intersection_area / union_area if union_area > 0 else 0.0

In [5]:
def non_max_suppression(boxes, scores, threshold=0.5):
    """
    Apply non-maximum suppression to eliminate redundant overlapping boxes

    Args:
        boxes: List of (x1, y1, x2, y2) bounding boxes
        scores: List of confidence scores for each box
        threshold: IoU threshold for suppression

    Returns:
        Lists of filtered boxes and scores
    """
    if len(boxes) == 0:
        return [], []

    # Convert to numpy arrays
    boxes = np.array(boxes)
    scores = np.array(scores)

    # Sort by score
    indices = np.argsort(scores)[::-1]
    boxes = boxes[indices]
    scores = scores[indices]

    keep = []

    while len(boxes) > 0:
        # Pick the box with highest score
        keep.append(0)

        if len(boxes) == 1:
            break

        # Calculate IoU with all other boxes
        ious = []
        for i in range(1, len(boxes)):
            ious.append(calculate_iou(boxes[0], boxes[i]))

        # Remove boxes that overlap too much
        indices = np.where(np.array(ious) < threshold)[0]
        indices = indices + 1  # Adjust indices (skip the first box)

        boxes = boxes[indices]
        scores = scores[indices]

    return boxes[keep].tolist(), scores[keep].tolist()

In [6]:
def calculate_iou(box1, box2):
    """Calculate Intersection over Union between two bounding boxes"""
    x1_1, y1_1, x2_1, y2_1 = box1
    x1_2, y1_2, x2_2, y2_2 = box2

    # Calculate intersection area
    x_left = max(x1_1, x1_2)
    y_top = max(y1_1, y1_2)
    x_right = min(x2_1, x2_2)
    y_bottom = min(y2_1, y2_2)

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)

    # Calculate union area
    box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
    box2_area = (x2_2 - x1_2) * (y2_2 - y1_2)
    union_area = box1_area + box2_area - intersection_area

    return intersection_area / union_area if union_area > 0 else 0.0

In [14]:
import cv2
import time
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.framework.formats import landmark_pb2

def process_video(input_path, model_path, output_folder="output", fps_target=None,
                  num_poses=3, min_pose_detection_confidence=0.5,
                  min_pose_presence_confidence=0.5, min_tracking_confidence=0.5):
    """
    Process a video file with multiple people dancing using MediaPipe Tasks API:
    1. Detect multiple poses simultaneously using the PoseLandmarker
    2. Create two output videos: original with overlay and black background

    Args:
        input_path (str): Path to input video file
        model_path (str): Path to the pose landmarker model (.task file)
        output_folder (str): Folder to save output videos
        fps_target (float, optional): Target FPS for output videos
        num_poses (int): Maximum number of people to detect in the video
        min_pose_detection_confidence (float): Minimum confidence for pose detection
        min_pose_presence_confidence (float): Minimum confidence for pose presence
        min_tracking_confidence (float): Minimum confidence for tracking
    """
    # Create output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Get input video filename without extension
    input_filename = os.path.splitext(os.path.basename(input_path))[0]
    timestamp = time.strftime("%Y%m%d-%H%M%S")

    # Paths for output videos
    output_overlay_path = os.path.join(output_folder, f"{input_filename}_pose_overlay_{timestamp}.mp4")
    output_black_path = os.path.join(output_folder, f"{input_filename}_pose_black_{timestamp}.mp4")

    # Initialize video capture
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {input_path}")
        return

    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Use target FPS if specified, otherwise use input video FPS
    output_fps = fps_target if fps_target is not None else fps

    # Initialize video writers
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_overlay = cv2.VideoWriter(output_overlay_path, fourcc, output_fps, (width, height))
    out_black = cv2.VideoWriter(output_black_path, fourcc, output_fps, (width, height))

    # Variables for tracking processing
    frame_count = 0
    start_time = time.time()
    last_progress_time = start_time

    # Create a shared variable to store processed frames
    processed_frames = {'overlay': None, 'black': None, 'timestamp': 0}

    def process_result(detection_result, output_image, timestamp_ms):
        """Callback function for processing pose detection results"""
        if timestamp_ms <= processed_frames['timestamp']:
            return

        # Update timestamp
        processed_frames['timestamp'] = timestamp_ms

        # Get RGB image from MediaPipe Image
        rgb_image = output_image.numpy_view()

        # Create black background
        black_image = np.zeros_like(rgb_image)

        # Draw landmarks on both images
        pose_landmarks_list = detection_result.pose_landmarks

        if pose_landmarks_list:
            # Create overlay image (copy of RGB image)
            overlay_image = np.copy(rgb_image)

            # Loop through the detected poses to visualize
            for idx in range(len(pose_landmarks_list)):
                pose_landmarks = pose_landmarks_list[idx]

                # Convert landmarks to proto format for drawing
                pose_landmarks_proto = landmark_pb2.NormalizedLandmarkList()
                pose_landmarks_proto.landmark.extend([
                    landmark_pb2.NormalizedLandmark(
                        x=landmark.x,
                        y=landmark.y,
                        z=landmark.z) for landmark in pose_landmarks
                ])

                # Draw on overlay image
                mp.solutions.drawing_utils.draw_landmarks(
                    overlay_image,
                    pose_landmarks_proto,
                    mp.solutions.pose.POSE_CONNECTIONS,
                    mp.solutions.drawing_styles.get_default_pose_landmarks_style()
                )

                # Draw on black background
                mp.solutions.drawing_utils.draw_landmarks(
                    black_image,
                    pose_landmarks_proto,
                    mp.solutions.pose.POSE_CONNECTIONS,
                    mp.solutions.drawing_styles.get_default_pose_landmarks_style()
                )

            # Store processed images
            processed_frames['overlay'] = cv2.cvtColor(overlay_image, cv2.COLOR_RGB2BGR)
            processed_frames['black'] = cv2.cvtColor(black_image, cv2.COLOR_RGB2BGR)
        else:
            # If no poses detected, just use the original frame for overlay
            processed_frames['overlay'] = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
            processed_frames['black'] = cv2.cvtColor(black_image, cv2.COLOR_RGB2BGR)

    # Setup MediaPipe Pose Landmarker
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.IMAGE,  # Use VIDEO mode instead of LIVE_STREAM for file processing
        num_poses=num_poses,
        min_pose_detection_confidence=min_pose_detection_confidence,
        min_pose_presence_confidence=min_pose_presence_confidence,
        min_tracking_confidence=min_tracking_confidence,
        output_segmentation_masks=False
    )

    print(f"Processing video: {input_path}")
    print(f"Total frames: {total_frames}")
    print(f"Resolution: {width}x{height}, FPS: {fps}")

    # Create the pose landmarker
    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        try:
            while cap.isOpened():
                # Read frame
                success, frame = cap.read()
                if not success:
                    break

                # Convert to RGB for MediaPipe
                mp_image = mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                )

                # Get timestamp in milliseconds
                timestamp_ms = int(frame_count * (1000 / fps))

                # Process frame with pose landmarker
                detection_result = landmarker.detect(mp_image)

                # Process the detection result manually (since we're using VIDEO mode)
                process_result(detection_result, mp_image, timestamp_ms)

                # Write frames to output videos if available
                if processed_frames['overlay'] is not None and processed_frames['black'] is not None:
                    out_overlay.write(processed_frames['overlay'])
                    out_black.write(processed_frames['black'])

                # Update progress
                frame_count += 1
                current_time = time.time()

                # Update progress every 2 seconds
                if current_time - last_progress_time > 2:
                    elapsed_time = current_time - start_time
                    fps_processing = frame_count / elapsed_time if elapsed_time > 0 else 0
                    progress = (frame_count / total_frames) * 100
                    num_detected = len(detection_result.pose_landmarks) if detection_result.pose_landmarks else 0

                    print(f"Progress: {progress:.1f}% ({frame_count}/{total_frames}), "
                          f"Processing speed: {fps_processing:.1f} FPS, "
                          f"People detected: {num_detected}")
                    last_progress_time = current_time

        finally:
            # Release resources
            cap.release()
            out_overlay.release()
            out_black.release()

            total_time = time.time() - start_time
            print(f"Processing completed in {total_time:.2f} seconds")
            print(f"Output videos saved to:")
            print(f"  - {output_overlay_path}")
            print(f"  - {output_black_path}")


# Alternative implementation using LIVE_STREAM mode
def process_video_live_stream(input_path, model_path, output_folder="output", fps_target=None,
                              num_poses=3, min_pose_detection_confidence=0.5,
                              min_pose_presence_confidence=0.5, min_tracking_confidence=0.5):
    """
    Alternative implementation using LIVE_STREAM mode with callback
    """
    # Create output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Get input video filename without extension
    input_filename = os.path.splitext(os.path.basename(input_path))[0]
    timestamp = time.strftime("%Y%m%d-%H%M%S")

    # Paths for output videos
    output_overlay_path = os.path.join(output_folder, f"{input_filename}_pose_overlay_{timestamp}.mp4")
    output_black_path = os.path.join(output_folder, f"{input_filename}_pose_black_{timestamp}.mp4")

    # Initialize video capture
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {input_path}")
        return

    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Use target FPS if specified, otherwise use input video FPS
    output_fps = fps_target if fps_target is not None else fps

    # Initialize video writers
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_overlay = cv2.VideoWriter(output_overlay_path, fourcc, output_fps, (width, height))
    out_black = cv2.VideoWriter(output_black_path, fourcc, output_fps, (width, height))

    # Variables for tracking processing
    frame_count = 0
    start_time = time.time()
    last_progress_time = start_time
    last_timestamp_ms = 0

    # Create a shared variable to store processed frames
    processed_frames = {'overlay': None, 'black': None}

    def process_result(detection_result, output_image, timestamp_ms):
        """Callback function for processing pose detection results"""
        nonlocal last_timestamp_ms

        # Skip if this is an older frame (can happen due to async nature)
        if timestamp_ms < last_timestamp_ms:
            return

        # Update timestamp
        last_timestamp_ms = timestamp_ms

        # Get RGB image from MediaPipe Image
        rgb_image = output_image.numpy_view()

        # Create black background
        black_image = np.zeros_like(rgb_image)

        # Draw landmarks on both images
        pose_landmarks_list = detection_result.pose_landmarks

        if pose_landmarks_list:
            # Create overlay image (copy of RGB image)
            overlay_image = np.copy(rgb_image)

            # Loop through the detected poses to visualize
            for idx in range(len(pose_landmarks_list)):
                pose_landmarks = pose_landmarks_list[idx]

                # Convert landmarks to proto format for drawing
                pose_landmarks_proto = landmark_pb2.NormalizedLandmarkList()
                pose_landmarks_proto.landmark.extend([
                    landmark_pb2.NormalizedLandmark(
                        x=landmark.x,
                        y=landmark.y,
                        z=landmark.z) for landmark in pose_landmarks
                ])

                # Draw on overlay image
                mp.solutions.drawing_utils.draw_landmarks(
                    overlay_image,
                    pose_landmarks_proto,
                    mp.solutions.pose.POSE_CONNECTIONS,
                    mp.solutions.drawing_styles.get_default_pose_landmarks_style()
                )

                # Draw on black background
                mp.solutions.drawing_utils.draw_landmarks(
                    black_image,
                    pose_landmarks_proto,
                    mp.solutions.pose.POSE_CONNECTIONS,
                    mp.solutions.drawing_styles.get_default_pose_landmarks_style()
                )

            # Store processed images
            processed_frames['overlay'] = cv2.cvtColor(overlay_image, cv2.COLOR_RGB2BGR)
            processed_frames['black'] = cv2.cvtColor(black_image, cv2.COLOR_RGB2BGR)
        else:
            # If no poses detected, just use the original frame for overlay
            processed_frames['overlay'] = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
            processed_frames['black'] = cv2.cvtColor(black_image, cv2.COLOR_RGB2BGR)

        # Write frames to output videos
        if processed_frames['overlay'] is not None and processed_frames['black'] is not None:
            out_overlay.write(processed_frames['overlay'])
            out_black.write(processed_frames['black'])

        # Update progress tracking
        nonlocal frame_count, last_progress_time
        frame_count += 1
        current_time = time.time()

        # Update progress every 2 seconds
        if current_time - last_progress_time > 2:
            elapsed_time = current_time - start_time
            fps_processing = frame_count / elapsed_time if elapsed_time > 0 else 0
            progress = (frame_count / total_frames) * 100
            num_detected = len(detection_result.pose_landmarks) if detection_result.pose_landmarks else 0

            print(f"Progress: {progress:.1f}% ({frame_count}/{total_frames}), "
                  f"Processing speed: {fps_processing:.1f} FPS, "
                  f"People detected: {num_detected}")
            last_progress_time = current_time

    # Setup MediaPipe Pose Landmarker with LIVE_STREAM mode
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.LIVE_STREAM,
        num_poses=num_poses,
        min_pose_detection_confidence=min_pose_detection_confidence,
        min_pose_presence_confidence=min_pose_presence_confidence,
        min_tracking_confidence=min_tracking_confidence,
        output_segmentation_masks=False,
        result_callback=process_result
    )

    print(f"Processing video: {input_path}")
    print(f"Total frames: {total_frames}")
    print(f"Resolution: {width}x{height}, FPS: {fps}")

    # Create the pose landmarker
    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        try:
            while cap.isOpened():
                # Read frame
                success, frame = cap.read()
                if not success:
                    break

                # Convert to RGB for MediaPipe
                mp_image = mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                )

                # Get timestamp in milliseconds
                timestamp_ms = int(frame_count * (1000 / fps))

                # Process frame with pose landmarker (async)
                landmarker.detect_async(mp_image, timestamp_ms)

                # Add a small delay to allow the callback to process
                time.sleep(0.001)

            # After reading all frames, allow time for final callbacks to complete
            time.sleep(1)

        finally:
            # Release resources
            cap.release()
            out_overlay.release()
            out_black.release()

            total_time = time.time() - start_time
            print(f"Processing completed in {total_time:.2f} seconds")
            print(f"Output videos saved to:")
            print(f"  - {output_overlay_path}")
            print(f"  - {output_black_path}")




# Funciones principales

In [8]:
def hand_detector_main():
    # Initialize webcam
    cap = cv2.VideoCapture(0)

    # Create hand detector
    detector = HandDetector(min_detection_confidence=0.7)

    # Variables for FPS calculation
    prev_time = 0
    current_time = 0

    while True:
        # Read frame from webcam
        success, img = cap.read()

        if not success:
            print("Failed to grab frame")
            break

        # Find hands
        img, hands = detector.find_hands(img)

        # Get landmark positions (optional)
        # landmark_list = detector.find_position(img, hand_index=0, draw=False)

        # Calculate and display FPS
        current_time = time.time()
        fps = 1 / (current_time - prev_time) if (current_time - prev_time) > 0 else 0
        prev_time = current_time

        # Display FPS
        cv2.putText(img, f'FPS: {int(fps)}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # Display hand information
        y_pos = 70
        for i, hand in enumerate(hands):
            hand_type = hand.get("type", "Unknown")
            cv2.putText(img, f'Hand {i+1}: {hand_type}', (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            y_pos += 30

            # Optionally display specific landmark information
            # For example, index finger tip (landmark 8)
            landmarks = hand.get("landmarks", [])
            for lm_id, x, y in landmarks:
                if lm_id == 8:  # Index finger tip
                    cv2.putText(img, f'Index Tip: ({x}, {y})', (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    y_pos += 30

        # Display the frame
        cv2.imshow('Hand Tracking', img)

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release resources
    cap.release()
    cv2.destroyAllWindows()

In [9]:
def pose_detector_main(input_path, output_path='results', fps_target=30):
    # Process the video
    process_video(input_path, output_path, fps_target, target_people=3)

# Ejecucion principal

In [18]:
input_video = "test2Video.mp4"
model_path = "pose_landmarker_full.task"
output_dir = "results"

In [19]:
# Process video
process_video(
    input_path=input_video,
    model_path=model_path,
    output_folder=output_dir,
    num_poses=4,  # Detect up to 4 people
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

Processing video: test2Video.mp4
Total frames: 96
Resolution: 1920x1080, FPS: 25.0


I0000 00:00:1744254759.828836   22557 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M3 Pro
W0000 00:00:1744254759.872400   80125 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1744254759.881113   80127 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Progress: 38.5% (37/96), Processing speed: 18.1 FPS, People detected: 3
Progress: 81.2% (78/96), Processing speed: 19.2 FPS, People detected: 1
Processing completed in 4.68 seconds
Output videos saved to:
  - results/test2Video_pose_overlay_20250409-211239.mp4
  - results/test2Video_pose_black_20250409-211239.mp4


In [21]:
hand_detector_main()
#pose_detector_main('IMG_9334.mp4')

2025-04-09 21:16:11.560 Python[3409:22557] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.
I0000 00:00:1744254972.724957   22557 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M3 Pro
W0000 00:00:1744254972.732164   86634 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1744254972.738748   86631 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
2025-04-09 21:16:13.151 Python[3409:22557] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-09 21:16:13.151 Python[3409:22557] +[IMKInputSession subclass]: chose IMKInputSession_Modern


KeyboardInterrupt: 